# 05 — Export to HuggingFace

**Wire a working model into the API.** The FastAPI service (`src/api/fastapi_main.py`) calls the HF Inference API at
`https://router.huggingface.co/hf-inference/models/{HF_MODEL}`.

Two routes:

## Option A — use the pre-trained BERTweet detector (fastest)

`04_evaluation.ipynb` evaluates `NLP-LTU/bertweet-large-sexism-detector`, which is already on the Hub. Point the API at it:

In [ ]:
REPO_ID = "NLP-LTU/bertweet-large-sexism-detector"

# In your .env:
#   HF_MODEL=NLP-LTU/bertweet-large-sexism-detector
#   HF_API_KEY=<your HF token with inference access>
#
# Then restart the API:
#   uvicorn src.api.fastapi_main:app --reload
#
# And test:
#   curl -X POST "http://localhost:8000/predict" #        -H "Content-Type: application/json" #        -d '{"text": "example text"}'

## Option B — export your own trained model

Uploads the TF-IDF + Logistic Regression baseline from `outputs/models/` (trained via `03_modeling.ipynb` or `python -m src.models.train`).

In [ ]:
from huggingface_hub import HfApi
from src.config.settings import HF_TOKEN, MODELS_DIR

MY_REPO = "your-username/sexism-classifier"  # <- change me

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=MY_REPO, repo_type="model", exist_ok=True)
api.upload_folder(
    repo_id=MY_REPO,
    repo_type="model",
    folder_path=str(MODELS_DIR),
    commit_message="Add TF-IDF + Logistic Regression sexism classifier",
)
print(f"Uploaded to https://huggingface.co/{MY_REPO}")

## Verify the Inference API works

In [ ]:
import requests
from src.config.settings import HF_API_KEY, HF_MODEL

r = requests.post(
    f"https://router.huggingface.co/hf-inference/models/{HF_MODEL}",
    headers={"Authorization": f"Bearer {HF_API_KEY}"},
    json={"inputs": "example text to classify"},
    timeout=30,
)
print(r.status_code, r.json())